<a href="https://colab.research.google.com/github/juanmareolon/OBL_ML/blob/main/Arbol_Clasificacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Árbol de clasificación — predicción de `opinion`

Obligatorio MLSI 2026 S2. Esta notebook contiene:

1. Carga de datos
2. **Preprocesamiento común** (idéntico en todas las notebooks de modelos)
3. *(próximo)* Búsqueda de hiperparámetros, evaluación y submission a Kaggle

## 1. Carga de datos

In [ ]:
import numpy as np
import pandas as pd

SEED = 42

# El repo es público: en Colab se leen los CSV directo desde GitHub.
# Si se ejecuta localmente con el repo clonado, se usan los archivos de data/.
import os
if os.path.exists('data/movies_train.csv'):
    BASE = 'data/'
else:
    print("No esta bajada la carpeta data, se leen los CSV desde GitHub")
    BASE = 'https://raw.githubusercontent.com/juanmareolon/OBL_ML/main/data/'

train = pd.read_csv(BASE + 'movies_train.csv')
test = pd.read_csv(BASE + 'movies_test.csv')

print('train:', train.shape, '| test:', test.shape)
train.head()

train: (3602, 14) | test: (1201, 14)


,budget,keywords,original_language,original_title,overview,popularity,release_date,revenue,runtime,status,vote_count,director,main_genre_top10,opinion
0,70000000,budapest kidnapping boxer secret agent liberat...,en,I Spy,"When the Switchblade, the most sophisticated p...",13.267631,2002-10-31,33561137,97.0,Released,269,Betty Thomas,Action,0
1,7,flooding futuristic police detective,en,Split Second,"In a futuristic London, the rising sea levels ...",4.857028,1992-05-01,5,90.0,Released,63,Tony Maylam,Thriller,0
2,14000000,college suspense gossip rumor social experiment,en,Gossip,"On a beautiful college campus, something ugly ...",5.833687,2000-04-21,5108820,90.0,Released,68,Davis Guggenheim,Drama,0
3,15000000,barcelona spain menage a trois author,en,Vicky Cristina Barcelona,Two girlfriends on a summer holiday in Spain b...,32.758254,2008-08-15,96408652,96.0,Released,1020,Woody Allen,Drama,1
4,250000000,witch magic broom school of witchcraft wizardry,en,Harry Potter and the Half-Blood Prince,"As Harry begins his sixth year at Hogwarts, he...",98.885637,2009-07-07,933959197,153.0,Released,5293,David Yates,Adventure,1


## 2. Preprocesamiento común

> ⚠️ **Bloque común: no modificar.** Este bloque se copia igual en todas las notebooks de modelos
> para que la comparación entre modelos sea justa. Las variantes experimentales van en una sección aparte.

El preprocesamiento tiene dos capas:

**Capa 1: limpieza fija (`limpiar`).** Transformaciones que no aprenden nada de los datos, así que se pueden aplicar una sola vez a train y test sin riesgo de *data leakage*:

| Columna | Decisión | Motivo |
|---|---|---|
| `budget`, `revenue` | 0 y valores < 1000 → NaN, más indicador `*_desconocido` | Los 0 son faltantes (21% y 29% de las filas) y los valores < 1000 son errores de unidad (ej. *Split Second*: budget = 7). Que falte el dato puede ser informativo. |
| `runtime` | 0 → NaN | Ninguna película dura 0 minutos. |
| `vote_count` | Se mantienen los 0 | Es un dato real: las 45 películas con 0 votos tienen `opinion` = 0. |
| `budget`, `revenue`, `vote_count`, `popularity` | `log1p` | Distribuciones muy sesgadas (ej. `popularity`: mediana 13, máximo 875). |
| `release_date` | Se extraen `anio` y `mes` | El año está muy asociado a `opinion` (pre-1980: 79% positivas; post-2005: 45%) y el mes también (temporada de premios a fin de año). |
| `keywords` | `sin_keywords` y `cant_keywords` | Las películas sin keywords tienen 31% de opinión positiva contra 49% del total. |
| nueva | `log_roi` = log1p(revenue / budget) | Relación recaudación/costo; solo si ambos datos existen. |
| `status`, `original_title`, `overview` | Se descartan | `status` es 99,8% "Released", el título es casi un id y `overview` queda para la parte de NLP. |

**Capa 2: pipeline de sklearn (`crear_preprocesador`).** Todo lo que *aprende* de los datos (medianas, escalas, categorías frecuentes, encodings) va dentro de un `ColumnTransformer`. Así, en cada fold de la validación cruzada se ajusta solo con el train de ese fold.

- **Numéricas:** imputación por mediana + `StandardScaler` (necesario para la regresión logística/lineal; no afecta a los árboles).
- **`main_genre_top10`:** one-hot.
- **`original_language`:** one-hot, agrupando los idiomas con menos de 20 películas en una categoría "infrecuente" (quedan inglés, francés, español y otros).
- **`director`** (1942 valores, 1275 con una sola película) con tres variantes para comparar:
  - `'ninguno'`: no se usa.
  - `'freq'`: frequency encoding (cantidad de películas del director en el train).
  - `'freq+target'`: frequency + target encoding con suavizado (`TargetEncoder`, que usa validación cruzada interna para no filtrar el target).

In [2]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder

TARGET = 'opinion'
UMBRAL_MONTO = 1000  # budget/revenue por debajo de esto se consideran faltantes


def limpiar(df):
    """Capa 1: limpieza fija (no aprende de los datos, se aplica igual a train y test)."""
    out = pd.DataFrame(index=df.index)

    # budget / revenue: 0 y valores absurdos (< 1000) -> NaN, con indicador de faltante
    for col in ['budget', 'revenue']:
        valor = df[col].where(df[col] >= UMBRAL_MONTO)
        out[f'{col}_desconocido'] = valor.isna().astype(int)
        out[f'log_{col}'] = np.log1p(valor)

    # ROI: solo si budget y revenue son conocidos
    budget = df['budget'].where(df['budget'] >= UMBRAL_MONTO)
    revenue = df['revenue'].where(df['revenue'] >= UMBRAL_MONTO)
    out['log_roi'] = np.log1p(revenue / budget)

    # runtime: 0 -> NaN
    out['runtime'] = df['runtime'].where(df['runtime'] > 0)

    # vote_count (los 0 son reales) y popularity: log1p
    out['log_vote_count'] = np.log1p(df['vote_count'])
    out['log_popularity'] = np.log1p(df['popularity'])

    # Fecha de estreno -> año y mes
    fecha = pd.to_datetime(df['release_date'], errors='coerce')
    out['anio'] = fecha.dt.year
    out['mes'] = fecha.dt.month

    # keywords: indicador de faltante y cantidad de palabras
    out['sin_keywords'] = df['keywords'].isna().astype(int)
    out['cant_keywords'] = df['keywords'].fillna('').str.split().str.len()

    # Categóricas (se codifican dentro del pipeline)
    out['main_genre_top10'] = df['main_genre_top10']
    out['original_language'] = df['original_language']
    out['director'] = df['director'].fillna('desconocido')
    return out


class FrequencyEncoder(BaseEstimator, TransformerMixin):
    """Reemplaza cada categoría por su cantidad de apariciones en el train (0 si no aparece)."""

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.columnas_ = list(X.columns)
        self.conteos_ = {c: X[c].value_counts() for c in self.columnas_}
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.columnas_)
        return np.column_stack([
            X[c].map(self.conteos_[c]).fillna(0).to_numpy(dtype=float) for c in self.columnas_
        ])

    def get_feature_names_out(self, input_features=None):
        return np.array([f'{c}_freq' for c in self.columnas_])


def crear_target_encoder():
    """TargetEncoder reproducible. Desde sklearn 1.9 la semilla se pasa vía `cv`;
    en versiones anteriores (Colab trae 1.6) vía `random_state`."""
    import sklearn
    from sklearn.model_selection import KFold
    version = tuple(int(v) for v in sklearn.__version__.split('.')[:2])
    if version >= (1, 9):
        return TargetEncoder(target_type='binary', cv=KFold(5, shuffle=True, random_state=SEED))
    return TargetEncoder(target_type='binary', random_state=SEED)


COLS_NUMERICAS = [
    'log_budget', 'log_revenue', 'log_roi', 'runtime', 'log_vote_count', 'log_popularity',
    'anio', 'mes', 'cant_keywords',
]
COLS_BINARIAS = ['budget_desconocido', 'revenue_desconocido', 'sin_keywords']


def crear_preprocesador(director='freq+target'):
    """Capa 2: ColumnTransformer que se ajusta dentro de cada fold.

    director: 'ninguno' | 'freq' | 'freq+target'
    """
    numericas = Pipeline([
        ('imputar', SimpleImputer(strategy='median')),
        ('escalar', StandardScaler()),
    ])
    transformadores = [
        ('num', numericas, COLS_NUMERICAS),
        ('bin', 'passthrough', COLS_BINARIAS),
        ('genero', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['main_genre_top10']),
        ('idioma', OneHotEncoder(min_frequency=20, handle_unknown='infrequent_if_exist',
                                 sparse_output=False), ['original_language']),
    ]
    if director in ('freq', 'freq+target'):
        transformadores.append(('director_freq', Pipeline([
            ('freq', FrequencyEncoder()),
            ('escalar', StandardScaler()),
        ]), ['director']))
    if director == 'freq+target':
        transformadores.append(('director_target', crear_target_encoder(), ['director']))
    elif director not in ('ninguno', 'freq'):
        raise ValueError(f'director inválido: {director}')

    return ColumnTransformer(transformadores, remainder='drop', verbose_feature_names_out=False)

### Aplicación del preprocesamiento

In [3]:
X = limpiar(train)
y = train[TARGET]
X_test = limpiar(test)
ids_test = test['id']

print('X:', X.shape, '| X_test:', X_test.shape)
print('Balance de clases:', y.value_counts(normalize=True).round(3).to_dict())
X.head()

X: (3602, 15) | X_test: (1201, 15)
Balance de clases: {0: 0.506, 1: 0.494}


,budget_desconocido,log_budget,revenue_desconocido,log_revenue,log_roi,runtime,log_vote_count,log_popularity,anio,mes,sin_keywords,cant_keywords,main_genre_top10,original_language,director
0,0,18.064006,0,17.328879,0.391667,97.0,5.598422,2.657993,2002.0,10.0,0,8,Action,en,Betty Thomas
1,1,NaN,1,NaN,NaN,90.0,4.158883,1.767642,1992.0,5.0,0,4,Thriller,en,Tony Maylam
2,0,16.454568,0,15.446479,0.311093,90.0,4.234107,1.921864,2000.0,4.0,0,6,Drama,en,Davis Guggenheim
3,0,16.523561,0,18.384107,2.005155,96.0,6.928538,3.519225,2008.0,8.0,0,6,Drama,en,Woody Allen
4,0,19.336971,0,20.654943,1.555158,153.0,8.574329,4.604026,2009.0,7.0,0,7,Adventure,en,David Yates


## 3. Árbol de clasificación

*(Próximo paso: definir los hiperparámetros a explorar, la validación cruzada y las métricas.)*